In [3]:
from __future__ import annotations

from pathlib import Path
from lib import text_spliter as ts
from sentence_transformers import SentenceTransformer
import chromadb
import uuid
from dotenv import load_dotenv

load_dotenv(verbose=True)

True

In [4]:
# model = SentenceTransformer("BAAI/bge-base-en-v1.5")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
client = chromadb.PersistentClient(path="chroma_db")
collection_name="matrix_minilm"
# collection = client.get_collection(collection_name)
if collection_name in [c.name for c in client.list_collections()]:
    client.delete_collection(name=collection_name)
collection = client.get_or_create_collection(
    name=collection_name, embedding_function=None, metadata={"hnsw:space": "cosine"}
)

In [6]:
def upsert_docs_to_chroma(docs):
    batch_size=256

    texts = [d["text"] for d in docs]
    metas = [d.get("meta", {}) for d in docs]
    ids = [str(uuid.uuid4()) for _ in metas]

    for start in range(0, len(texts), batch_size):
        end = start + batch_size
        batch_texts = texts[start:end]
        batch_metas = metas[start:end]
        batch_ids = ids[start:end]

        # embeddings: list[list[float]] (dim=384 for MiniLM-L6)
        batch_emb = model.encode(
            batch_texts,
            batch_size=64,
            show_progress_bar=False,
            normalize_embeddings=True,  # cosine-friendly
        ).tolist()

        collection.upsert(
            ids=batch_ids,
            documents=batch_texts,
            metadatas=batch_metas,
            embeddings=batch_emb,
        )

    return collection


In [7]:
from pathlib import Path
from tqdm.notebook import tqdm

dst_dir = Path("knowledge_base")
files = list(dst_dir.glob("*.md"))

for p in tqdm(files, desc="Indexing markdown files"):
    title = p.stem.replace("_", " ")
    text = p.read_text(encoding="utf-8", errors="strict")
    upsert_docs_to_chroma(ts.make_embedding_docs(text, title))

print(f"Processed {len(files)} files")


Processed 23 files


In [9]:
q = "What relations between Olezeq and YanOCHka?"
# q = "Who is Olezeq enemy?"

# q = "Describe Yan04ka"
# q = "Who is 0lezeq?"
# q = "What relations between 0lezeq and Yan04ka?"
# q = "What relations between 0lezeq and Yan04ka?"
# q = "Who love 0lezeq?"
# q = "Who love Yan04ka"
# q = "Where 0lezeq come from?"
# q = "What real name of 0lezeq?"
# q = "Who is V@sya Pupk1n?"
# q = "What is Binarywood?"
q_emb = model.encode([q], normalize_embeddings=True).tolist()
res = collection.query(
    query_embeddings=q_emb,
    n_results=10,
    include=["documents", "metadatas", "distances"],
)

for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
    print("dist", dist)
    print("meta")
    print(meta)
    print("doc")
    print(doc[:500], "...")
    print("\n---\n")

# print(len(res["documents"][0]))

dist 0.2883158326148987
meta
{'section_path': 'The VectOr Reloaded : The Resistance : Return to Binarywood', 'title': 'Olezeq', 'chunk_index': 1, 'end_line': 459, 'start_line': 443}
doc
Title: Olezeq
Section: The VectOr Reloaded > The Resistance > Return to Binarywood

anOCHka use the brief respite to express their passion for one another. For when the elevator doors open once again, Olezeq and YanOCHka are greeted by the sight of a sea of followers presenting gifts for Olezeq and pleading for him to watch over their loved ones. YanOCHka goes on ahead and leaves Olezeq to assume his role as The Perv1y, promising to Olezeq that there would be time.

YanOCHka and Olezeq passionat ...

---

dist 0.3045993447303772
meta
{'chunk_index': 0, 'title': 'Olezeq', 'end_line': 94, 'section_path': 'Family : Romances', 'start_line': 92}
doc
Title: Olezeq
Section: Family > Romances

YanOCHka ...

---

dist 0.31517088413238525
meta
{'title': 'YanOCHka', 'start_line': 55, 'section_path': 'Family : Roma